# IMPORT LIBRARIES

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import isnull, col, sum, count, round, countDistinct, lower
from pyspark.sql.functions import regexp_replace, length, lpad
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

spark = SparkSession.builder \
    .appName("Olist Project") \
    .getOrCreate()

## cek duplikasi & null vallue buat olist_orders_database

In [2]:
# olist_orders_database cek
df_orders = spark.read.csv(
    "hdfs://localhost:8020/user/cloudera/olist/raw/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
) 

In [3]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [4]:
# df.describe biasa
df_orders.describe().toPandas()

,summary,order_id,customer_id,order_status
0,count,99441,99441,99441
1,mean,None,None,None
2,stddev,None,None,None
3,min,00010242fe8c5a6d1ba2dd792cb16214,00012a2ce6f8dcda20d059ce98491703,approved
4,max,fffe41c64501cc87c801fd61db3f6244,ffffe8b65bbe3087b653a978c870db99,unavailable


In [5]:
# cek null
cek_null_orders = df_orders.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_orders.columns
])

cek_null_orders.toPandas().T

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


In [6]:
# persentase null
total_orders = df_orders.count()

persentase_null_orders = df_orders.select([
    round((sum(col(c).isNull().cast("int")) / total_orders * 100), 3).alias(c)
    for c in df_orders.columns
])

persentase_null_orders.toPandas().T

,0
order_id,0.000
customer_id,0.000
order_status,0.000
order_purchase_timestamp,0.000
order_approved_at,0.161
order_delivered_carrier_date,1.793
order_delivered_customer_date,2.982
order_estimated_delivery_date,0.000


In [7]:
#cek duplikasi
total_rows_orders = df_orders.count()
unique_rows_orders = df_orders.dropDuplicates().count()

print("Total rows:", total_rows_orders)
print("Unique rows:", unique_rows_orders)
print("Duplicate rows:", total_rows_orders - unique_rows_orders)

Total rows: 99441
Unique rows: 99441
Duplicate rows: 0


In [8]:
# cek uniqueness key
UKO1 = df_orders.select("order_id").distinct().count()
UKO2 = df_orders.select("customer_id").distinct().count()

print(UKO1)
print(UKO2)

99441
99441


In [9]:
# deteksi anomali waktu: Tanggal (approved) terjadi sebelum tanggal pembelian (purchase)
anomaly_approval_df = df_orders.filter(
    col("order_approved_at") < col("order_purchase_timestamp")
)
anomaly_approval_count = anomaly_approval_df.count()

print(f"Total anomali disetujui sebelum dibeli : {anomaly_approval_count}")


Total anomali disetujui sebelum dibeli : 0


In [10]:
# deteksi anomali waktu: Tanggal diterima pelanggan terjadi sebelum tanggal pembelian
anomaly_delivery_df = df_orders.filter(
    col("order_delivered_customer_date") < col("order_purchase_timestamp")
)
anomaly_delivery_count = anomaly_delivery_df.count()

print(f"Total anomali diterima sebelum dibeli : {anomaly_delivery_count}\n")

Total anomali diterima sebelum dibeli : 0



In [11]:
# hapus missing value

df_orders = df_orders.dropna(subset=[
    "order_purchase_timestamp",
    "order_delivered_customer_date"
])

In [12]:
# hapus duplicated

df_orders = df_orders.dropDuplicates()

In [89]:
# perlakuan left semi join biar gk ada missing relation

count_payments_before = df_order_payments.count()
df_order_payments = df_order_payments.join(df_orders, on="order_id", how="left_semi")
print(f"[Payments] sebelum: {count_payments_before} | Sesudah: {df_order_payments.count()}")

count_items_before = df_order_items.count()
df_order_items = df_order_items.join(df_orders, on="order_id", how="left_semi")
print(f"[Items] sebelum: {count_items_before} | Sesudah: {df_order_items.count()}")

count_reviews_before = df_order_reviews.count()
df_order_reviews = df_order_reviews.join(df_orders, on="order_id", how="left_semi")
print(f"[Reviews] sebelum: {count_reviews_before} | Sesudah: {df_order_reviews.count()}")

count_customers_before = df_customers.count()
df_customers = df_customers.join(df_orders, on="customer_id", how="left_semi")
print(f"[Customers] sebelum: {count_customers_before} | Sesudah: {df_customers.count()}")

[Payments] sebelum: 103886 | Sesudah: 100754
[Items] sebelum: 112650 | Sesudah: 110196
[Reviews] sebelum: 99224 | Sesudah: 96359
[Customers] sebelum: 99441 | Sesudah: 96476


## cek duplikasi & null value buat olist_customers_dataset

In [15]:
# olist_products_dataset cek
df_customers = spark.read.csv(
    "hdfs://localhost:8020/user/cloudera/olist/raw/olist_customers_dataset.csv",
    header = True,
    inferSchema = True
)

In [16]:
df_customers = df_customers.withColumn(
    "customer_zip_code_prefix", 
    lpad(col("customer_zip_code_prefix").cast("string"), 5, "0")
)

In [17]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [18]:
# df.describe biasa
df_customers.describe().toPandas()

,summary,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,count,99441,99441,99441,99441,99441
1,mean,None,None,35137.47458291851,None,None
2,stddev,None,None,29797.938996206158,None,None
3,min,00012a2ce6f8dcda20d059ce98491703,0000366f3b9a7992bf8c76cfdf3221e2,01003,abadia dos dourados,AC
4,max,ffffe8b65bbe3087b653a978c870db99,ffffd2657e2aad2907e67c3e9daecbeb,99990,zortea,TO


In [19]:
# cek null
cek_null_customers = df_customers.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_customers.columns
])

cek_null_customers.toPandas().T

,0
customer_id,0
customer_unique_id,0
customer_zip_code_prefix,0
customer_city,0
customer_state,0


In [20]:
# persentase null
total_customers = df_customers.count()

persentase_null_customers = df_customers.select([
    round((sum(col(c).isNull().cast("int")) / total_customers * 100), 3).alias(c)
    for c in df_customers.columns
])

persentase_null_customers.toPandas().T

,0
customer_id,0.0
customer_unique_id,0.0
customer_zip_code_prefix,0.0
customer_city,0.0
customer_state,0.0


In [21]:
# cek duplikasi
total_rows_customers = df_customers.count()
unique_rows_customers = df_customers.dropDuplicates().count()

print("Total rows:", total_rows_customers)
print("Unique rows:", unique_rows_customers)
print("Duplicate rows:", total_rows_customers - unique_rows_customers)

Total rows: 99441
Unique rows: 99441
Duplicate rows: 0


In [22]:
# cek uniqueness key
UKC1 = df_customers.select("customer_id").distinct().count()
UKC2 = df_customers.select("customer_unique_id").distinct().count()
UKC3 = df_customers.select("customer_zip_code_prefix").distinct().count()

print(UKC1)
print(UKC2)
print(UKC3)

99441
96096
14994


In [23]:
# validasi keunikan customer_id
unique_customer_id = df_customers.select("customer_id").distinct().count()
print(f"total baris customer row: {total_customers}")
print(f"total customer_id unik: {unique_customer_id}")

if total_customers == unique_customer_id:
    print("valid, setaip order mpy customer_id yg unik")
else:
    print("ada duplikasi")

total baris customer row: 99441
total customer_id unik: 99441
valid, setaip order mpy customer_id yg unik


In [24]:
# hitung pelanggan asli
unique_real_cust = df_customers.select("customer_unique_id").distinct().count()
print(f"total pelanggan asli: {unique_real_cust}")

total pelanggan asli: 96096


In [25]:
# deteksi pelanggan yg repurchase
repurchase = df_customers.groupBy("customer_unique_id") \
    .agg(count("customer_id").alias("total_orders")) \
    .filter(col("total_orders") > 1) \
    .orderBy(col("total_orders").desc())

total_repurchasers = repurchase.count()
print(f"Total pelanggan yang repurchase: {total_repurchasers} orang\n")

Total pelanggan yang repurchase: 2997 orang



In [26]:
# display top 5 pelanggan
repurchase.show(5, truncate=False)

+--------------------------------+------------+
|customer_unique_id              |total_orders|
+--------------------------------+------------+
|8d50f5eadf50201ccdcedfb9e2ac8455|17          |
|3e43e6105506432c953e165fb2acf44c|9           |
|1b6c7548a2a1f9037c1fd3ddfed95f33|7           |
|6469f99c1f9dfae7733b25662e7f1782|7           |
|ca77025e7201e3b30c44b472ff346268|7           |
+--------------------------------+------------+
only showing top 5 rows



In [27]:
# validasi ulang zip_code_prefix

# 1. Mendeteksi kode pos yang panjangnya kurang dari 5 digit
invalid_zip_df = df_customers.filter(length(col("customer_zip_code_prefix")) < 5)
invalid_zip_count = invalid_zip_df.count()

print(f"Total kode pos dengan format terpotong (< 5 digit): {invalid_zip_count}\n")

if invalid_zip_count > 0:
    print("Contoh kode pos yang terpotong (Kehilangan angka 0 di depan):")
    invalid_zip_df.select("customer_id", "customer_zip_code_prefix").show(5, truncate=False)

Total kode pos dengan format terpotong (< 5 digit): 0



In [28]:
# data cleaning: nambahin angka 0 depan biar pas 5 digit

if invalid_zip_count > 0:
    print("Contoh kode pos yang terpotong (Kehilangan angka 0 di depan):")
    invalid_zip_df.select("customer_id", "customer_zip_code_prefix").show(5, truncate=False)
    
    df_customers = df_customers.withColumn(
        "customer_zip_code_prefix",
        lpad(col("customer_zip_code_prefix"), 5, "0")
    )
    
    print("Contoh data setelah dibersihkan (Leading Zero dikembalikan):")
    df_customers.join(invalid_zip_df.select("customer_id"), "customer_id", "inner") \
        .select("customer_id", "customer_zip_code_prefix") \
        .show(5, truncate=False)
        
else:
    print("Semua kode pos sudah memiliki format minimal 5 digit.")

Semua kode pos sudah memiliki format minimal 5 digit.


## cek duplikasi & null value buat olist_order_items_dataset

In [29]:
# load data
df_order_items = spark.read.csv(
    "hdfs://localhost:8020/user/cloudera/olist/raw/olist_order_items_dataset.csv",
    header = True,
    inferSchema = True
)

In [30]:
# display schema
df_order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [31]:
# df.describe biasa
df_order_items.describe().toPandas()

,summary,order_id,order_item_id,product_id,seller_id,price,freight_value
0,count,112650,112650,112650,112650,112650,112650
1,mean,None,1.1978339991122948,None,None,120.65373901471354,19.99031992898364
2,stddev,None,0.7051240313951722,None,None,183.63392805026012,15.806405412297071
3,min,00010242fe8c5a6d1ba2dd792cb16214,1,00066f42aeeb9f3007548bb9d3f33c38,0015a82c2db000af6aaaf3ae2ecb0532,0.85,0.0
4,max,fffe41c64501cc87c801fd61db3f6244,21,fffe9eeff12fcbd74a2f2b007dde0c58,ffff564a4f9085cd26170f4732393726,6735.0,409.68


In [32]:
# cek null
cek_null_order_items = df_order_items.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_order_items.columns
])
cek_null_order_items.toPandas().T

,0
order_id,0
order_item_id,0
product_id,0
seller_id,0
shipping_limit_date,0
price,0
freight_value,0


In [33]:
# persentase cek null
total_order_items = df_order_items.count()

persentase_null_order_items = df_order_items.select([
    round((sum(col(c).isNull().cast("int")) / total_order_items * 100), 3).alias(c)
    for c in df_order_items.columns
])

persentase_null_order_items.toPandas().T

,0
order_id,0.0
order_item_id,0.0
product_id,0.0
seller_id,0.0
shipping_limit_date,0.0
price,0.0
freight_value,0.0


In [34]:
# cek duplikasi
total_rows_order_items = df_order_items.count()
unique_rows_order_items = df_order_items.dropDuplicates().count()

print("Total rows: ", total_rows_order_items)
print("Unique rows: ", unique_rows_order_items)
print("Duplicate rows: ", total_rows_order_items - unique_rows_order_items)

Total rows:  112650
Unique rows:  112650
Duplicate rows:  0


In [35]:
# cek uniqueness key
UKOI1 = df_order_items.select("order_id").distinct().count()
UKOI2 = df_order_items.select("order_item_id").distinct().count()
UKOI3 = df_order_items.select("product_id").distinct().count()
UKOI4 = df_order_items.select("seller_id").distinct().count()

print(UKOI1)
print(UKOI2)
print(UKOI3)
print(UKOI4)

98666
21
32951
3095


In [36]:
# composite key check
df_order_items.select("order_id", "order_item_id").dropDuplicates().count()

112650

## cek duplikasi & null value buat olist_order_reviews_dataset

In [37]:
df_order_reviews = spark.read.csv(
    "hdfs://localhost:8020/user/cloudera/olist/raw/olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True
)

In [38]:
# display schema
df_order_reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [39]:
# df.describe biasa
df_order_reviews.describe().toPandas()

,summary,review_id,order_id,review_score,review_comment_title,review_comment_message
0,count,99224,99224,99224,11568,40977
1,mean,None,None,4.08642062404257,3.165434995880696E10,8.172413793103448
2,stddev,None,None,1.3475791311150984,5.625434750908947E11,3.1175650470615843
3,min,0001239bc1de2e33cb583967c2ca4c67,00010242fe8c5a6d1ba2dd792cb16214,1,,\r\n
4,max,fffefe7a48d22f7b32046421062219d1,fffe41c64501cc87c801fd61db3f6244,5,🔟,😡😡😡😡😡👎👎👎👎👎\r\nEmpresa sem compromisso com o cl...


In [40]:
# cek null
cek_null_order_reviews = df_order_reviews.select([
    sum(col(c).isNull().cast('int')).alias(c)
    for c in df_order_reviews.columns
])
cek_null_order_reviews.toPandas().T

,0
review_id,0
order_id,0
review_score,0
review_comment_title,87656
review_comment_message,58247
review_creation_date,0
review_answer_timestamp,0


In [41]:
# persentase cek null
total_order_reviews = df_order_reviews.count()

persentase_null_order_reviews = df_order_reviews.select([
    round((sum(col(c).isNull().cast("int")) / total_order_reviews * 100), 3).alias(c)
    for c in df_order_reviews.columns
])

persentase_null_order_reviews.toPandas().T

,0
review_id,0.000
order_id,0.000
review_score,0.000
review_comment_title,88.342
review_comment_message,58.703
review_creation_date,0.000
review_answer_timestamp,0.000


In [42]:
# cek duplikasi
total_rows_order_reviews = df_order_reviews.count()
unique_rows_order_reviews = df_order_reviews.dropDuplicates().count()

print("Total rows: ", total_rows_order_reviews)
print("Unique rows: ", unique_rows_order_reviews)
print("Duplicate rows:", total_rows_order_reviews - unique_rows_order_reviews)

Total rows:  99224
Unique rows:  99224
Duplicate rows: 0


In [43]:
# cek uniqueness key
UKOR1 = df_order_reviews.select("review_id").distinct().count()
UKOR2 = df_order_reviews.select("order_id").distinct().count()

print(UKOR1)
print(UKOR2)

98410
98673


In [44]:
df_order_reviews.select("review_score").distinct().show()

+------------+
|review_score|
+------------+
|           1|
|           3|
|           5|
|           4|
|           2|
+------------+



In [45]:
# value validation
df_order_reviews.toPandas().T

,0,1,2,3,4,5,6,7,8,9,...,99214,99215,99216,99217,99218,99219,99220,99221,99222,99223
review_id,7bc2406110b926393aa56f80a40eba40,80e641a11e56f04c1ad469d5645fdfde,228ce5500dc1d8e020d8d1322874b6f0,e64fb393e7b32834bb789ff8bb30750e,f7c4243c7fe1938f181bec41a392bdeb,15197aa66ff4d0650b5434f1b46cda19,07f9bee5d1b850860defd761afa7ff16,7c6400515c67679fbee952a7525281ef,a3f6f7f6f433de0aefbb97da197c554c,8670d52e15e00043ae7de4c01cc2fe06,...,3c569245c7610d9a51eff50032194ea3,a709d176f59bc3af77f4149c96bae357,115d72425fc59eec9a2e9d7594c6d93d,c6b270c61f67c9f7cb07d84ea8aeaf8b,af2dc0519de6e0720ef0c74292fb4114,574ed12dd733e5fa530cfd4bbf39d7c9,f3897127253a9592a73be9bdfdf4ed7a,b3de70c89b1510c4cd3d0649fd302472,1adeb9d84d72fe4e337617733eb85149,efe49f1d6f951dd88b51e6ccd4cc548f
order_id,73fc7af87114b39712e6da79b0a377eb,a548910a1c6147796b98fdf73dbeba33,f9e4b658b201a9f2ecdecbb34bed034b,658677c97b385a9be170737859d3511b,8e6bfb81e283fa7e4f11123a3fb894f1,b18dcdf73be66366873cd26c5724d1dc,e48aa0d2dcec3a2e87348811bcfdf22b,c31a859e34e3adac22f376954e19b39d,9c214ac970e84273583ab523dfafd09b,b9bf720beb4ab3728760088589c62129,...,bf25ff262495b0b876bc4f594f736ecf,d5cb12269711bd1eaf7eed8fd32a7c95,acd45245723df7cb52772a34416b41b1,48f7ee67313eda32bfcf5b9c1dd9522d,d699c734a0b1c8111f2272a3f36d398c,2a8c23fee101d4d5662fa670396eb8da,22ec9f0669f784db00fa86d035cf8602,55d4004744368f5571d1f590031933e4,7725825d039fc1f0ceb7635e3f7d9206,90531360ecb1eec2a1fbb265a0db0508
review_score,4,5,5,5,5,1,5,5,5,4,...,5,3,3,5,5,5,5,5,4,1
review_comment_title,None,None,None,None,None,None,None,None,None,recomendo,...,None,None,None,None,None,None,None,None,None,None
review_comment_message,None,None,None,Recebi bem antes do prazo estipulado.,Parabéns lojas lannister adorei comprar pela I...,None,None,None,None,aparelho eficiente. no site a marca do aparelh...,...,None,"O produto não foi enviado com NF, não existe v...",None,None,None,None,None,"Excelente mochila, entrega super rápida. Super...",None,"meu produto chegou e ja tenho que devolver, po..."
review_creation_date,2018-01-18 00:00:00,2018-03-10 00:00:00,2018-02-17 00:00:00,2017-04-21 00:00:00,2018-03-01 00:00:00,2018-04-13 00:00:00,2017-07-16 00:00:00,2018-08-14 00:00:00,2017-05-17 00:00:00,2018-05-22 00:00:00,...,2018-05-23 00:00:00,2018-05-19 00:00:00,2018-05-08 00:00:00,2017-12-13 00:00:00,2018-04-27 00:00:00,2018-07-07 00:00:00,2017-12-09 00:00:00,2018-03-22 00:00:00,2018-07-01 00:00:00,2017-07-03 00:00:00
review_answer_timestamp,2018-01-18 21:46:59,2018-03-11 03:05:13,2018-02-18 14:36:24,2017-04-21 22:02:06,2018-03-02 10:26:53,2018-04-16 00:39:37,2017-07-18 19:30:34,2018-08-14 21:36:06,2017-05-18 12:05:37,2018-05-23 16:45:47,...,2018-05-26 21:24:06,2018-05-20 21:51:06,2018-05-10 14:37:03,2017-12-14 11:09:36,2018-04-30 01:18:57,2018-07-14 17:18:30,2017-12-11 20:06:42,2018-03-23 09:10:43,2018-07-02 12:59:13,2017-07-03 21:01:49


In [46]:
# cleaning nama samaran Game Of Throne

# 1. Daftar klan Game of Thrones
got_houses = ["stark", "lannister", "targaryen", "baratheon", "greyjoy", 
              "martell", "tyrell", "arryn", "tully", "frey", "bolton"]

# Membuat pola regex dari daftar klan
got_pattern = "|".join(got_houses)

In [47]:
# 2. Mendeteksi ulasan yang mengandung nama-nama klan Game of Thrones
# fokus pada kolom review_comment_message yang tidak null
reviews_with_text_df = df_order_reviews.filter(col("review_comment_message").isNotNull())

anonymized_reviews_df = reviews_with_text_df.filter(
    lower(col("review_comment_message")).rlike(got_pattern)
)

anonymized_count = anonymized_reviews_df.count()
print(f"Total ulasan yang mengandung nama samaran: {anonymized_count} baris\n")

if anonymized_count > 0:
    print("Contoh ulasan dengan nama samaran (Sebelum dibersihkan):")
    anonymized_reviews_df.select("review_id", "review_comment_message")\
    .show(5, truncate=False)

Total ulasan yang mengandung nama samaran: 1950 baris

Contoh ulasan dengan nama samaran (Sebelum dibersihkan):
+--------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|review_id                       |review_comment_message                                                                                                                                                                                     |
+--------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|f7c4243c7fe1938f181bec41a392bdeb|Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa                                     

In [48]:
# 3. Data Cleaning: Mengganti nama klan dengan kata yang lebih netral 
# mengganti ke kata "loja" toko dalam bahasa portugis agar 
# tidak merusak analisis NLP

df_reviews_cleaned = reviews_with_text_df.withColumn(
    "review_comment_message_clean",
    regexp_replace(lower(col("review_comment_message")), got_pattern, "[loja]")
)

print("Contoh ulasan setelah nama samaran dibersihkan:")
df_reviews_cleaned.filter(lower(col("review_comment_message")).rlike(got_pattern)) \
    .select("review_comment_message", "review_comment_message_clean") \
    .show(5, truncate=False)

Contoh ulasan setelah nama samaran dibersihkan:
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|review_comment_message                                                                                                                                                                                     |review_comment_message_clean                                                                                                                                                                            |
+-----------------------------------------------------------------------------------------------------------------------------------------

## cek duplikasi & null value buat olist_order_payments_dataset

In [49]:
# load data
df_order_payments = spark.read.csv(
    "hdfs://localhost:8020/user/cloudera/olist/raw/olist_order_payments_dataset.csv",
    header = True,
    inferSchema = True
)

In [50]:
# display schema
df_order_payments.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



In [51]:
# df.describe biasa
df_order_payments.describe().toPandas()

,summary,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,count,103886,103886,103886,103886,103886
1,mean,None,1.0926785129853878,None,2.853348863176944,154.10038041698587
2,stddev,None,0.7065837791949938,None,2.6870506738564925,217.49406386472404
3,min,00010242fe8c5a6d1ba2dd792cb16214,1,boleto,0,0.0
4,max,fffe41c64501cc87c801fd61db3f6244,29,voucher,24,13664.08


In [52]:
# cek null
cek_null_order_payments = df_order_payments.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_order_payments.columns
])
cek_null_order_payments.toPandas().T

,0
order_id,0
payment_sequential,0
payment_type,0
payment_installments,0
payment_value,0


In [53]:
#persentase cek null
total_order_payments = df_order_payments.count()

persentase_null_order_payments = df_order_payments.select([
    round((sum(col(c).isNull().cast("int")) / total_order_payments * 100), 3).alias(c)
    for c in df_order_payments.columns
])
persentase_null_order_payments.toPandas().T

,0
order_id,0.0
payment_sequential,0.0
payment_type,0.0
payment_installments,0.0
payment_value,0.0


In [54]:
# cek duplikasi
total_rows_order_payments = df_order_payments.count()
unique_rows_order_payments = df_order_payments.dropDuplicates().count()

print("Total rows: ", total_rows_order_payments)
print("Unique rows: ", unique_rows_order_payments)
print("Duplicated rows: ", total_rows_order_payments - unique_rows_order_payments)

Total rows:  103886
Unique rows:  103886
Duplicated rows:  0


In [55]:
# cek uniqueness key
df_order_payments.select("order_id").distinct().count()

99440

## cek duplikasi & null value buat olist_product_dataset

In [56]:
df_products = spark.read.csv(
    "hdfs://localhost:8020/user/cloudera/olist/raw/olist_products_dataset.csv",
    header = True,
    inferSchema = True
)

In [57]:
df_products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [58]:
df_products.describe().toPandas()

,summary,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,count,32951,32341,32341,32341,32341,32949,32949,32949,32949
1,mean,None,None,48.47694876472589,771.4952846232337,2.1889861166939797,2276.4724877841513,30.81507784758263,16.937661234028347,23.196728277034204
2,stddev,None,None,10.245740725237287,635.1152246349538,1.7367656379315435,4282.038730977024,16.914458054065953,13.637554061749569,12.079047453227794
3,min,00066f42aeeb9f3007548bb9d3f33c38,agro_industria_e_comercio,5,4,1,0,7,2,6
4,max,fffe9eeff12fcbd74a2f2b007dde0c58,utilidades_domesticas,76,3992,20,40425,105,105,118


In [59]:
# cek null
cek_null_products = df_products.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_products.columns
])
cek_null_products.toPandas().T

,0
product_id,0
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,2
product_length_cm,2
product_height_cm,2
product_width_cm,2


In [60]:
# persentase cek null
total_products = df_products.count()

persentase_null_products = df_products.select([
    round((sum(col(c).isNull().cast("int")) / total_products * 100), 3).alias(c)
    for c in df_products.columns
])

persentase_null_products.toPandas().T

,0
product_id,0.000
product_category_name,1.851
product_name_lenght,1.851
product_description_lenght,1.851
product_photos_qty,1.851
product_weight_g,0.006
product_length_cm,0.006
product_height_cm,0.006
product_width_cm,0.006


In [61]:
# cek duplikasi
total_rows_products = df_products.count()
unique_rows_products = df_products.dropDuplicates().count()

print("Total rows: ", total_rows_products)
print("Unique rows: ", unique_rows_products)
print("Duplicate rows: ", total_rows_products - unique_rows_products)

Total rows:  32951
Unique rows:  32951
Duplicate rows:  0


In [62]:
df_products.select("product_id").distinct().count()

32951

## cek duplikasi & null value buat olist_geolocation_dataset

In [63]:
df_geolocation = spark.read.csv(
    "hdfs://localhost:8020/user/cloudera/olist/raw/olist_geolocation_dataset.csv",
    header = True,
    inferSchema = True
)

In [64]:
df_geolocation = df_geolocation.withColumn(
    "geolocation_zip_code_prefix", 
    lpad(col("geolocation_zip_code_prefix").cast("string"), 5, "0")
)

In [65]:
#printSchema
df_geolocation.printSchema()

root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



In [66]:
# df.describe biasa
df_geolocation.describe().toPandas()

,summary,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,count,1000163,1000163,1000163,1000163,1000163
1,mean,36574.16646586607,-21.176152910383045,-46.3905413209364,None,None
2,stddev,30549.335710319552,5.715866308823013,4.26974830661959,None,None
3,min,01001,-36.6053744107061,-101.46676644931476,* cidade,AC
4,max,99990,45.06593318269697,121.10539381057764,óleo,TO


In [67]:
# cek null
cek_null_geolocation = df_geolocation.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_geolocation.columns
])
cek_null_geolocation.toPandas().T

,0
geolocation_zip_code_prefix,0
geolocation_lat,0
geolocation_lng,0
geolocation_city,0
geolocation_state,0


In [68]:
# persentase cek null
total_geolocation = df_geolocation.count()

persentase_null_geolocation = df_geolocation.select([
    round((sum(col(c).isNull().cast("int")) / total_geolocation * 100), 3).alias(c)
    for c in df_geolocation.columns
])
persentase_null_geolocation.toPandas().T

,0
geolocation_zip_code_prefix,0.0
geolocation_lat,0.0
geolocation_lng,0.0
geolocation_city,0.0
geolocation_state,0.0


In [69]:
# cek duplikasi
total_rows_geolocation = df_geolocation.count()
unique_rows_geolocation = df_geolocation.dropDuplicates().count()

print("Total rows: ", total_rows_geolocation)
print("Unique rows: ", unique_rows_geolocation)
print("Duplicate: ", total_rows_geolocation - unique_rows_geolocation)

Total rows:  1000163
Unique rows:  738332
Duplicate:  261831


In [70]:
# cek uniqueness key
df_geolocation.select("geolocation_zip_code_prefix").distinct().count()

19015

In [71]:
# 1. Mendeteksi kode pos yang panjangnya kurang dari 5 digit
invalid_zip_df3 = df_geolocation.filter(length(col("geolocation_zip_code_prefix")) < 5)
invalid_zip_count3 = invalid_zip_df.count()

print(f"Total kode pos dengan format terpotong (< 5 digit): {invalid_zip_count3}\n")

if invalid_zip_count3 > 0:
    print("Contoh kode pos yang terpotong (Kehilangan angka 0 di depan):")
    invalid_zip_df3.select("geolocation_zip_code_prefix").show(5, truncate=False)

Total kode pos dengan format terpotong (< 5 digit): 0



In [72]:
if invalid_zip_count3 > 0:
    print("Contoh kode pos yang terpotong (Kehilangan angka 0 di depan):")
    invalid_zip_df3.select("geolocation_zip_code_prefix").show(5, truncate=False)
    df_geolocation = df_geolocation.withColumn(
        "geolocation_zip_code_prefix",
        lpad(col("geolocation_zip_code_prefix"), 5, "0")
    )
    
    print("Contoh data setelah dibersihkan (Leading Zero dikembalikan):")
    df_geolocation.join("inner") \
        .select("geolocation_id", "geolocation_zip_code_prefix") \
        .show(5, truncate=False)
        
else:
    print("Semua kode pos sudah memiliki format minimal 5 digit.")

Semua kode pos sudah memiliki format minimal 5 digit.


## cek duplikasi & null value buat olist_sellers_dataset

In [73]:
df_sellers = spark.read.csv(
    "hdfs://localhost:8020/user/cloudera/olist/raw/olist_sellers_dataset.csv",
    header = True,
    inferSchema = True,
)

In [74]:
df_sellers = df_sellers.withColumn(
    "seller_zip_code_prefix", 
    lpad(col("seller_zip_code_prefix").cast("string"), 5, "0")
)

In [75]:
df_sellers.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [76]:
df_sellers.describe().show()

+-------+--------------------+----------------------+-----------+------------+
|summary|           seller_id|seller_zip_code_prefix|seller_city|seller_state|
+-------+--------------------+----------------------+-----------+------------+
|  count|                3095|                  3095|       3095|        3095|
|   mean|                null|    32291.059450726978|  4482255.0|        null|
| stddev|                null|     32713.45382950901|        NaN|        null|
|    min|0015a82c2db000af6...|                 01001|   04482255|          AC|
|    max|ffff564a4f9085cd2...|                 99730|      xaxim|          SP|
+-------+--------------------+----------------------+-----------+------------+



In [77]:
# cek null
cek_null_sellers = df_sellers.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_sellers.columns
])
cek_null_sellers.toPandas().T

,0
seller_id,0
seller_zip_code_prefix,0
seller_city,0
seller_state,0


In [78]:
# persentase cek null
total_sellers = df_sellers.count()

persentase_null_sellers = df_sellers.select([
    round((sum(col(c).isNull().cast("int")) / total_sellers * 100), 3).alias(c)
    for c in df_sellers.columns
])
persentase_null_sellers.toPandas().T

,0
seller_id,0.0
seller_zip_code_prefix,0.0
seller_city,0.0
seller_state,0.0


In [79]:
# cek duplikasi
total_rows_sellers = df_sellers.count()
unique_rows_sellers = df_sellers.dropDuplicates().count()

print("Total rows: ", total_rows_sellers)
print("Unique rows: ", unique_rows_sellers)
print("Duplicates: ", total_rows_sellers - unique_rows_sellers)

Total rows:  3095
Unique rows:  3095
Duplicates:  0


In [80]:
# cek uniqueness key
df_sellers.select("seller_zip_code_prefix").distinct().count()

2246

In [81]:
# 1. Mendeteksi kode pos yang panjangnya kurang dari 5 digit
invalid_zip_df2 = df_sellers.filter(length(col("seller_zip_code_prefix")) < 5)
invalid_zip_count2 = invalid_zip_df2.count()

print(f"Total kode pos dengan format terpotong (< 5 digit): {invalid_zip_count2}\n")

if invalid_zip_count2 > 0:
    print("Contoh kode pos yang terpotong (Kehilangan angka 0 di depan):")
    invalid_zip_df2.select("seller_id", "seller_zip_code_prefix").show(5, truncate=False)

Total kode pos dengan format terpotong (< 5 digit): 0



In [82]:
# data cleaning

if invalid_zip_count2 > 0:
    print("Contoh kode pos yang terpotong (Kehilangan angka 0 di depan):")
    invalid_zip_df2.select("seller_id", "seller_zip_code_prefix").show(5, truncate=False)
    df_sellers = df_sellers.withColumn(
        "seller_zip_code_prefix",
        lpad(col("seller_zip_code_prefix"), 5, "0")
    )
    
    print("Contoh data setelah dibersihkan (Leading Zero dikembalikan):")
    df_sellers.join(invalid_zip_df2.select("seller_id"), "seller_id", "inner") \
        .select("seller_id", "seller_zip_code_prefix") \
        .show(5, truncate=False)
        
else:
    print("Semua kode pos sudah memiliki format minimal 5 digit.")

Semua kode pos sudah memiliki format minimal 5 digit.


## cek duplikasi & null value buat olist_PCNT_dataset

In [83]:
# load data
df_PCNT = spark.read.csv(
    "hdfs://localhost:8020/user/cloudera/olist/raw/product_category_name_translation.csv",
    header = True,
    inferSchema = True
)

In [84]:
df_PCNT.printSchema()

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



In [85]:
df_PCNT.describe().show()

+-------+---------------------+-----------------------------+
|summary|product_category_name|product_category_name_english|
+-------+---------------------+-----------------------------+
|  count|                   71|                           71|
|   mean|                 null|                         null|
| stddev|                 null|                         null|
|    min| agro_industria_e_...|         agro_industry_and...|
|    max| utilidades_domest...|                watches_gifts|
+-------+---------------------+-----------------------------+



In [86]:
# cek null
cek_null_PCNT = df_PCNT.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_PCNT.columns
])
cek_null_PCNT.toPandas().T

,0
product_category_name,0
product_category_name_english,0


In [87]:
# persentase cek null
total_PCNT = df_PCNT.count()

persentase_null_PCNT = df_PCNT.select([
    round((sum(col(c).isNull().cast("int")) / total_PCNT * 100), 3).alias(c)
    for c in df_PCNT.columns
])
persentase_null_PCNT.toPandas().T

,0
product_category_name,0.0
product_category_name_english,0.0


In [88]:
# cek duplikasi
total_rows_PCNT = df_PCNT.count()
unique_rows_PCNT = df_PCNT.dropDuplicates().count()

print("Total rows: ", total_rows_PCNT)
print("Unique rows: ", unique_rows_PCNT)
print("Duplicate rows: ", total_rows_PCNT - unique_rows_PCNT)

Total rows:  71
Unique rows:  71
Duplicate rows:  0


## Referential Integrity Check

In [90]:
df_orders.join(df_customers, "customer_id", "left_anti").count()

0

In [91]:
# fungsi bantuan buat cek RI
def referential_integrity(child_df, parent_df, child_key, parent_key, child_name, parent_name):
    orphan_df = child_df.join(
        parent_df,
        child_df[child_key] == parent_df[parent_key],
        how = "left_anti"
    )
    
    orphan_count = orphan_df.count()

    print(f"[{child_name}] -> [{parent_name}] (Key: {child_key})")
    if orphan_count == 0:
        print("Status referensi valid")
    else:
        print(f"Ditemukan ada {orphan_count} baris tanpa referensi")

In [92]:
# payment ke orders (key: order_id)
referential_integrity(df_order_payments, df_orders, "order_id", "order_id"
                      , "Payments", "Orders")

[Payments] -> [Orders] (Key: order_id)
Status referensi valid


In [93]:
# reviews ke orders (key: order_id)
referential_integrity(df_order_reviews, df_orders, "order_id", "order_id"
                     , "Reviews", "Orders")

[Reviews] -> [Orders] (Key: order_id)
Status referensi valid


In [94]:
# order items ke order (key: order_id)
referential_integrity(df_order_items, df_orders, "order_id", "order_id"
                      , "Order Items", "Orders")

[Order Items] -> [Orders] (Key: order_id)
Status referensi valid


In [95]:
# order ke customers (key: customer_id)
referential_integrity(df_orders, df_customers, "customer_id", "customer_id"
                      , "Orders", "Customers")

[Orders] -> [Customers] (Key: customer_id)
Status referensi valid


In [96]:
# order item ke product (key: product_id)
referential_integrity(df_order_items, df_products, "product_id", "product_id"
                      , "Order Items", "Products")

[Order Items] -> [Products] (Key: product_id)
Status referensi valid


In [97]:
# order item ke sellers (key: seller_id)
referential_integrity(df_order_items, df_sellers, "seller_id", "seller_id"
                      , "Order Items", "Sellers")

[Order Items] -> [Sellers] (Key: seller_id)
Status referensi valid


In [98]:
# customer ke geolocation (key: zip_code_prefix)
referential_integrity(df_customers, df_geolocation, "customer_zip_code_prefix"
                      , "geolocation_zip_code_prefix", "Customers", "Geolocation")

[Customers] -> [Geolocation] (Key: customer_zip_code_prefix)
Ditemukan ada 264 baris tanpa referensi


In [99]:
# seller ke geolocation (key: zip_code_prefix)
referential_integrity(df_sellers, df_geolocation, "seller_zip_code_prefix"
                      , "geolocation_zip_code_prefix", "Sellers", "Geolocation")

[Sellers] -> [Geolocation] (Key: seller_zip_code_prefix)
Ditemukan ada 7 baris tanpa referensi


## validasi hubungan order_id dan order_item_id

In [100]:
# 1. Validasi Keunikan Composite Key (order_id + order_item_id)
total_rows_items = df_order_items.count()
unique_composite_keys = df_order_items.select("order_id", "order_item_id").distinct().count()

print(f"Total baris data order items            : {total_rows_items}")
print(f"Total kombinasi order_id & item_id unik : {unique_composite_keys}")

if total_rows_items == unique_composite_keys:
    print("VALID: Kombinasi 'order_id' dan 'order_item_id' tidak memiliki duplikasi.\n")
else:
    print(f"Ditemukan {total_rows_items - unique_composite_keys} baris duplikasi pada kombinasi kunci!\n")

Total baris data order items            : 110196
Total kombinasi order_id & item_id unik : 110196
VALID: Kombinasi 'order_id' dan 'order_item_id' tidak memiliki duplikasi.



In [101]:
# 2. Analisis Pesanan dengan Banyak Item dan Banyak Penjual

# Mengelompokkan berdasarkan order_id untuk melihat jumlah 
# item dan jumlah penjual per pesanan
order_complexity_df = df_order_items.groupBy("order_id").agg(
    count("order_item_id").alias("total_items"),
    countDistinct("seller_id").alias("total_unique_sellers")
)

# Mendeteksi pesanan yang melibatkan lebih dari 1 penjual (seller_id)
multi_seller_df = order_complexity_df.filter(col("total_unique_sellers") > 1). \
orderBy(col("total_unique_sellers").desc())

multi_seller_count = multi_seller_df.count()

print(f"Total order dari 1 penjual berbeda: {multi_seller_count} pesanan\n")

if multi_seller_count > 0:
    print("Contoh 5 pesanan teratas yang melibatkan penjual paling banyak:")
    multi_seller_df.show(5, truncate=False)

Total order dari 1 penjual berbeda: 1275 pesanan

Contoh 5 pesanan teratas yang melibatkan penjual paling banyak:
+--------------------------------+-----------+--------------------+
|order_id                        |total_items|total_unique_sellers|
+--------------------------------+-----------+--------------------+
|cf5c8d9f52807cb2d2f0a0ff54c478da|6          |5                   |
|1c11d0f4353b31ac3417fbfa5f0f2a8a|7          |5                   |
|91be51c856a90d7efe86cf9d082d6ae3|4          |4                   |
|8c2b13adf3f377c8f2b06b04321b0925|4          |4                   |
|1d23106803c48c391366ff224513fb7f|4          |4                   |
+--------------------------------+-----------+--------------------+
only showing top 5 rows



## outlier kgk semua, cuman yg dibutuhin buat modelling

In [102]:
def outlier_iqr(df, column_name):
    quantiles = df.approxQuantile(column_name, [0.25, 0.75], 0.01)
    q1 = quantiles[0]
    q3 = quantiles[1]
    
    iqr = q3 - q1
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)
    
    total_rows = df.count()
    
    outliers_df = df.filter(
        (col(column_name) < lower_bound) | 
        (col(column_name) > upper_bound)
    )
    
    outliers_count = outliers_df.count()
    
    outliers_percentage = (outliers_count / total_rows) * 100 if total_rows > 0 else 0
    
    print(f"Deteksi Outlier: Kolom '{column_name}'")
    print(f"Q1 (25%)    : {q1}")
    print(f"Q3 (75%)    : {q3}")
    print(f"IQR         : {iqr}")
    print(f"Batas Bawah : {lower_bound}")
    print(f"Batas Atas  : {upper_bound}")
    print(f"Total Data  : {total_rows} baris")
    print(f"Jml Outlier : {outliers_count} baris ({outliers_percentage:.2f}% dari total)\n")
    
    return outliers_df

In [103]:
outlier_payment = outlier_iqr(df_order_payments, "payment_value")

Deteksi Outlier: Kolom 'payment_value'
Q1 (25%)    : 55.69
Q3 (75%)    : 169.61
IQR         : 113.92000000000002
Batas Bawah : -115.19000000000003
Batas Atas  : 340.49
Total Data  : 100754 baris
Jml Outlier : 7780 baris (7.72% dari total)



In [104]:
outlier_price = outlier_iqr(df_order_items, "price")

Deteksi Outlier: Kolom 'price'
Q1 (25%)    : 39.9
Q3 (75%)    : 129.99
IQR         : 90.09
Batas Bawah : -95.23499999999999
Batas Atas  : 265.125
Total Data  : 110196 baris
Jml Outlier : 8573 baris (7.78% dari total)



In [105]:
outlier_freight = outlier_iqr(df_order_items, "freight_value")

Deteksi Outlier: Kolom 'freight_value'
Q1 (25%)    : 12.9
Q3 (75%)    : 21.03
IQR         : 8.13
Batas Bawah : 0.7050000000000001
Batas Atas  : 33.225
Total Data  : 110196 baris
Jml Outlier : 11808 baris (10.72% dari total)



## capping dari outlier

In [106]:
def cap_outlier(df, column_name):
    quantiles = df.approxQuantile(column_name, [0.25, 0.75], 0.01)
    q1 = quantiles[0]
    q3 = quantiles[1]
    
    # 2. Menghitung IQR dan menentukan Batas Bawah & Atas
    iqr = q3 - q1
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)
    
    print(f"[{column_name}] Proses Capping...")
    print(f"-> Nilai > {upper_bound:.2f} diubah menjadi {upper_bound:.2f}")
    print(f"-> Nilai < {lower_bound:.2f} diubah menjadi {lower_bound:.2f}\n")
    
    # 3. Menerapkan logika Capping
    df_capped = df.withColumn(
        column_name,
        F.when(F.col(column_name) > upper_bound, upper_bound)
         .when(F.col(column_name) < lower_bound, lower_bound)
         .otherwise(F.col(column_name))
    )
    
    return df_capped

In [107]:
df_order_payments = cap_outlier(df_order_payments, "payment_value")

df_order_items = cap_outlier(df_order_items, "price")

df_order_items = cap_outlier(df_order_items, "freight_value")

[payment_value] Proses Capping...
-> Nilai > 340.49 diubah menjadi 340.49
-> Nilai < -115.19 diubah menjadi -115.19

[price] Proses Capping...
-> Nilai > 265.12 diubah menjadi 265.12
-> Nilai < -95.23 diubah menjadi -95.23

[freight_value] Proses Capping...
-> Nilai > 33.23 diubah menjadi 33.23
-> Nilai < 0.71 diubah menjadi 0.71



## feature engineering & aggregation

In [109]:
# setup target classification (dari df_order_reviews)
# ubah review_score jadi biner: 1 puas, 2 gk puas

df_reviews_feat = df_order_reviews.withColumn(
    "is_satisfied",
    F.when(F.col("review_score") >= 4, 1).otherwise(0)
)
df_reviews_agg = df_reviews_feat.groupBy("order_id").agg(
    F.avg("is_satisfied").alias("is_satisfied")
)
df_reviews_agg = df_reviews_agg.withColumn(
    "is_satisfied",
    F.when(F.col("is_satisfied") >= 0.5, 1).otherwise(0)
)
print("Target 'is_satisfied' berhasil dibuat.")

Target 'is_satisfied' berhasil dibuat.


In [110]:
df_reviews_agg.show(5)

+--------------------+------------+
|            order_id|is_satisfied|
+--------------------+------------+
|e239d280236cdd3c4...|           1|
|d451da9b109e1786f...|           1|
|0de5dbc9f32267616...|           1|
|251f0a3981c4a8cb8...|           1|
|f63a31c3349b87273...|           1|
+--------------------+------------+
only showing top 5 rows



In [111]:
# feature insight 1 & 2: delivery time & late delivery (dari df_orders)

#pastikan kolom tanggal bertime timestamp
df_orders["order_purchase_timestamp", \
          "order_delivered_customer_date", "order_estimated_delivery_date"].printSchema()

#hitung selisih waktu dan keterlambatan
df_orders_feat = df_orders.withColumn(
    "delivery_time", 
    F.datediff(F.col("order_delivered_customer_date"), F.col("order_purchase_timestamp"))
).withColumn(
    "is_late", 
    F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1)\
    .otherwise(0)
)

#ambil kolom yg dibutuhin aja buat master table
df_orders_clean = df_orders_feat.select("order_id", "customer_id", "delivery_time", "is_late")
print("Fitur 'delivery_time' dan 'is_late' berhasil dibuat.")

root
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

Fitur 'delivery_time' dan 'is_late' berhasil dibuat.


In [112]:
df_orders_clean.show(5)

+--------------------+--------------------+-------------+-------+
|            order_id|         customer_id|delivery_time|is_late|
+--------------------+--------------------+-------------+-------+
|615ae1b88a25a4ac4...|25daddfce277eafad...|           10|      0|
|4d2d8a4224215f680...|5957d9ae822bcbe88...|           35|      1|
|12c35cf2a5171b0fb...|96d3fe655ac136a35...|            4|      0|
|11210d2ba14b85282...|51293a462a75bbaa6...|           29|      0|
|e93fceb3b0a859a71...|8f11b4d19fddeb5a1...|           12|      0|
+--------------------+--------------------+-------------+-------+
only showing top 5 rows



In [113]:
# feature insight 3: payment value (dari df_order_payments)
# 1 order bisa dibayar dgn beberapa metode
# harus agregasi (sum) biar totalnya jadi 1 baris per order_id

df_payments_agg = df_order_payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("total_payment_value")
)
print("Fitur 'total_payment_value' berhasil diagregasi.")


Fitur 'total_payment_value' berhasil diagregasi.


In [114]:
df_payments_agg.show(5)

+--------------------+-------------------+
|            order_id|total_payment_value|
+--------------------+-------------------+
|85be7c94bcd3f908f...|              72.75|
|8ca5bdac5ebe8f2d6...| 189.07999999999998|
|54066aeaaf3ac32e7...|             148.06|
|5db54d41d5ebd6d76...|             136.26|
|41537821ce113ccef...|              89.27|
+--------------------+-------------------+
only showing top 5 rows



In [115]:
# feature insight 4: order complexity (dari df_order_items)
# 1 order bisa ada byk item, itung jumalh baris item per order
# jumlahkan juga price & freight_value yg udh di capping tadi

df_items_agg = df_order_items.groupBy("order_id").agg(
    F.count("order_item_id").alias("total_items"),
    F.sum("price").alias("total_price"),
    F.sum("freight_value").alias("total_freight")
)
print("Fitur 'total_items' (Order Complexity) berhasil diagregasi.")

Fitur 'total_items' (Order Complexity) berhasil diagregasi.


In [116]:
df_items_agg.show(5)

+--------------------+-----------+-----------+-------------+
|            order_id|total_items|total_price|total_freight|
+--------------------+-----------+-----------+-------------+
|014405982914c2cde...|          2|      49.23|         29.2|
|019886de8f385a39b...|          1|      159.9|         28.5|
|01a6ad782455876aa...|          1|      34.99|         15.1|
|01d907b3e209269e1...|          1|     151.99|        17.77|
|028dc52e12ddda803...|          1|      49.99|        11.73|
+--------------------+-----------+-----------+-------------+
only showing top 5 rows



In [117]:
# feature insight 5: customer location (dari df_customers)

df_customers_clean = df_customers.select("customer_id", "customer_state", "customer_city")
print("Fitur lokasi customer disiapkan.")

Fitur lokasi customer disiapkan.


In [118]:
df_customers_clean.show(5)

+--------------------+--------------+--------------------+
|         customer_id|customer_state|       customer_city|
+--------------------+--------------+--------------------+
|06b8999e2fba1a1fb...|            SP|              franca|
|18955e83d337fd6b2...|            SP|sao bernardo do c...|
|4e7b3e00288586ebd...|            SP|           sao paulo|
|b2b6027bc5c5109e5...|            SP|     mogi das cruzes|
|4f2d8ab171c80ec83...|            SP|            campinas|
+--------------------+--------------+--------------------+
only showing top 5 rows



## Data Integration

In [119]:
# gabung semua tabel jadi 1 tabel gede
# pake left join dgn df_order_clean sbg lead table

df_master = df_orders_clean \
    .join(df_customers_clean, on="customer_id", how="left") \
    .join(df_reviews_agg, on="order_id", how="left") \
    .join(df_payments_agg, on="order_id", how="left") \
    .join(df_items_agg, on="order_id", how="left")

In [120]:
df_master.describe().toPandas().T

,0,1,2,3,4
summary,count,mean,stddev,min,max
order_id,96476,None,None,00010242fe8c5a6d1ba2dd792cb16214,fffe41c64501cc87c801fd61db3f6244
customer_id,96476,None,None,00012a2ce6f8dcda20d059ce98491703,ffffa3172527f765de70084a7e53aae8
delivery_time,96476,12.497336125046644,9.555459598617153,0,210
is_late,96476,0.08112898544715784,0.27303451407502777,0,1
customer_state,96476,None,None,AC,TO
customer_city,96476,None,None,abadia dos dourados,zortea
is_satisfied,95830,0.7899405196702494,0.4073527054310869,0,1
total_payment_value,96475,133.12786815237092,93.53153374759214,9.59,1360.49
total_items,96476,1.1422115344748953,0.5388157789144618,1,21


In [121]:
# data cleaning dikit
# apus pesanan yg gk ada target 'is_satisfied', soalnya gk bisa latih model tanpa label

df_master = df_master.dropna(subset=["is_satisfied"])

In [122]:
# isi nilai null dgn 0 utk fitur numerik hasil agregasi 
#(jaga2 biar aman kalo ada join yg gk ketemu)

df_master = df_master.fillna(0, subset=["total_payment_value", \
                                        "total_items", "total_price", "total_freight"])

In [123]:
df_master.describe().toPandas()

,summary,order_id,customer_id,delivery_time,is_late,customer_state,customer_city,is_satisfied,total_payment_value,total_items,total_price,total_freight
0,count,95830,95830,95830,95830,95830,95830,95830,95830,95830,95830,95830
1,mean,None,None,12.455838463946572,0.07995408535949076,None,None,0.7899405196702494,133.0260198267765,1.1412814358760304,110.96008483773377,20.317185902118368
2,stddev,None,None,9.469888507564018,0.2712235189397383,None,None,0.4073527054310869,93.48961476206931,0.5340974680642783,92.55661977711893,12.44946158376447
3,min,00010242fe8c5a6d1ba2dd792cb16214,00012a2ce6f8dcda20d059ce98491703,0,0,AC,abadia dos dourados,0,0.0,1,0.85,0.7050000000000001
4,max,fffe41c64501cc87c801fd61db3f6244,ffffa3172527f765de70084a7e53aae8,208,1,TO,zortea,1,1360.49,21,2651.25,365.4750000000001


## memuat file data akhir ke hdfs

In [125]:
# Simpan sebagai single file agar mudah di-download

df_master.coalesce(1).write.mode("append").parquet("hdfs://localhost:8020//user/cloudera/olist/processed")

#df_master.coalesce(1).write.mode("overwrite").csv("hdfs://localhost:8020//user/cloudera/olist/processed")